# 05 — Document segmentation, language detection and text preprocessing

This notebook prepares the PDF collection for downstream NLP tasks by extracting page-level text, identifying document sections, detecting document and section languages, and generating optional stop-word-filtered text.

It builds on the inventory and data-quality outputs created by the earlier notebooks. The original PDFs and existing metadata observations remain unchanged.

The preprocessing pipeline focuses on English and Dutch scientific documents. It preserves original text and protects metadata fields such as titles, author names, affiliations, keywords and identifiers from stop-word removal.

## Objectives

- Extract and organize text from the existing PDF collection.
- Identify document sections using reproducible, rule-based methods.
- Detect English, Dutch and potentially mixed-language content.
- Generate optional stop-word-filtered versions of eligible text.
- Preserve document identifiers and source information.
- Identify records requiring manual review.
- Produce quality summaries and visualizations.
- Export structured outputs for downstream tokenization and metadata extraction.

## Data sources

The notebook reads the existing inventory files from `../output/inventory/` and PDF documents from `../pdfs/`.

The `document_id` field is used to connect preprocessing results to the existing inventory and metadata tables.

## Reproducibility and privacy

Processing is performed locally using Python and open-source libraries. No document text is transmitted to external language-detection services.

Original PDF files, extracted text and existing metadata are not modified by this notebook. Generated outputs are stored separately under `../output/preprocessing/`.

## Execution

Run this notebook from the `notebooks/` directory, after the PDF collection inventory has been generated.

## 1. Dependencies and paths

Import the Python libraries required for PDF preprocessing and define the locations of the PDF collection, inventory files, and preprocessing outputs.

Run this notebook from the `notebooks/` folder, the same as the previous notebooks.

The required input directories and inventory file are checked before continuing. A separate output directory is created to store preprocessing results without modifying the original data.

In [ ]:
from pathlib import Path

import pandas as pd
import pymupdf
import matplotlib.pyplot as plt

PDF_DIR = Path("../pdfs").resolve()
INVENTORY_DIR = Path("../output/inventory").resolve()
OUTPUT_DIR = Path("../output/preprocessing").resolve()

if not PDF_DIR.is_dir():
    raise FileNotFoundError(f"PDF folder not found: {PDF_DIR}")

documents_path = INVENTORY_DIR / "documents.csv"

if not documents_path.is_file():
    raise FileNotFoundError(
        "Inventory missing. Run Notebook 01 first."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

documents = pd.read_csv(
    documents_path,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

print("PDF directory:", PDF_DIR)
print("Inventory directory:", INVENTORY_DIR)
print("Preprocessing output:", OUTPUT_DIR)
print("Documents loaded:", len(documents))

display(
    documents[
        ["document_id", "file_name", "page_count", "text_status"]
    ].head()
)

## 2. Inventory validation and extraction readiness

Check the completeness and consistency of the existing document inventory before preprocessing.

Verify that document identifiers are unique, PDF files are accessible, and extraction statuses are recorded. Summarize the documents that may require additional processing or manual review.

Visualize the distribution of text extraction statuses to assess how much of the collection is ready for segmentation and language detection.

In [ ]:
# Check required inventory columns
required_columns = {
    "document_id",
    "file_name",
    "pdf_path",
    "page_count",
    "text_status"
}

missing_columns = required_columns - set(documents.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

# Check document identifiers
duplicate_ids = documents["document_id"].duplicated().sum()
missing_ids = documents["document_id"].eq("").sum()

# Check that each PDF is accessible
documents_check = documents.copy()

documents_check["pdf_exists"] = documents_check["pdf_path"].apply(
    lambda path: (PDF_DIR / path).is_file()
)

# Summarize inventory validation
print("Total documents:", len(documents_check))
print("Duplicate document IDs:", duplicate_ids)
print("Missing document IDs:", missing_ids)
print("PDF files found:", documents_check["pdf_exists"].sum())
print("Missing PDF files:", (~documents_check["pdf_exists"]).sum())

if duplicate_ids or missing_ids:
    raise ValueError("Inventory contains invalid document identifiers.")

if not documents_check["pdf_exists"].all():
    display(
        documents_check.loc[
            ~documents_check["pdf_exists"],
            ["document_id", "file_name", "pdf_path"]
        ]
    )
    raise FileNotFoundError("Some inventory PDFs are missing.")

print("\nText extraction status:")
display(
    documents_check["text_status"]
    .value_counts()
    .rename_axis("status")
    .to_frame("documents")
)

In [ ]:
# Count documents by extraction status
status_counts = documents_check["text_status"].value_counts()

# Create visualization
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(
    status_counts.index,
    status_counts.values,
    color=["#579AF5", "#57C9A0", "#F29B85"][:len(status_counts)]
)

# Add document counts above bars
for bar in bars:
    height = bar.get_height()
    ax.annotate(
        f"{int(height)}",
        (bar.get_x() + bar.get_width() / 2, height),
        ha="center",
        va="bottom",
        xytext=(0, 4),
        textcoords="offset points"
    )

ax.set_title("PDF Text Extraction Readiness")
ax.set_xlabel("Text extraction status")
ax.set_ylabel("Number of documents")
ax.set_ylim(0, max(status_counts.values) * 1.15)

plt.tight_layout()
plt.show()

## 3. Page-level PDF text extraction

Extract text from each PDF page using PyMuPDF while retaining document identifiers and page numbers.

Preserve the original extracted text without applying cleaning or stop-word removal. Empty pages are retained so that partially extractable documents can be identified.

Save the page-level results separately from the existing inventory for use in document segmentation and language detection.

In [ ]:
# Extract text from every page of the PDF collection
page_records = []
extraction_errors = []

for row in documents_check.itertuples(index=False):
    pdf_path = PDF_DIR / row.pdf_path

    try:
        with pymupdf.open(pdf_path) as pdf:
            for page_index, page in enumerate(pdf):
                original_text = page.get_text("text", sort=True)

                page_records.append({
                    "document_id": row.document_id,
                    "file_name": row.file_name,
                    "page_number": page_index + 1,
                    "original_text": original_text,
                    "character_count": len(original_text),
                    "has_text": bool(original_text.strip())
                })

    except Exception as exc:
        extraction_errors.append({
            "document_id": row.document_id,
            "file_name": row.file_name,
            "error": f"{type(exc).__name__}: {exc}"
        })

# Convert extracted records to a DataFrame
pages_df = pd.DataFrame(
    page_records,
    columns=[
        "document_id",
        "file_name",
        "page_number",
        "original_text",
        "character_count",
        "has_text"
    ]
)

errors_df = pd.DataFrame(
    extraction_errors,
    columns=["document_id", "file_name", "error"]
)

print("Documents in inventory:", len(documents_check))
print("Pages extracted:", len(pages_df))
print("Pages containing text:", int(pages_df["has_text"].sum()))
print("Pages without text:", int((~pages_df["has_text"]).sum()))
print("PDF extraction errors:", len(errors_df))

display(
    pages_df[
        ["document_id", "file_name", "page_number",
         "character_count", "has_text"]
    ].head(10)
)

In [ ]:
# Check that extracted page counts match the inventory
expected_pages = pd.to_numeric(
    documents_check["page_count"],
    errors="coerce"
).sum()

actual_pages = len(pages_df)

print("Expected pages:", int(expected_pages))
print("Extracted pages:", actual_pages)

if errors_df.empty and actual_pages != expected_pages:
    raise ValueError(
        "Extracted page count does not match the inventory."
    )

# Save original page-level text as JSON Lines
pages_output = OUTPUT_DIR / "pages.jsonl"

pages_df.to_json(
    pages_output,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved page-level text to:", pages_output)

# Display extraction errors, if any
if not errors_df.empty:
    display(errors_df)

## 4. Document section segmentation

Identify common scientific document sections using English and Dutch heading patterns.

Separate recognized sections such as abstracts, introductions, methods, results, discussions, conclusions and references while preserving the original text and page information.

Unrecognized content is retained as `unknown` rather than assigned an uncertain section label. The results will be reviewed before language detection and stop-word filtering.

In [ ]:
import re

# Common English and Dutch scientific section headings
SECTION_PATTERNS = {
    "abstract": [
        r"abstract",
        r"samenvatting",
        r"managementsamenvatting",
        r"executive summary"
    ],
    "introduction": [
        r"introduction",
        r"inleiding",
        r"introductie",
        r"achtergrond"
    ],
    "methods": [
        r"materials? and methods?",
        r"methods?",
        r"methodology",
        r"methodologie",
        r"methode",
        r"methoden",
        r"onderzoeksopzet",
        r"onderzoeksaanpak"
    ],
    "results": [
        r"results?",
        r"resultaten",
        r"bevindingen",
        r"onderzoeksresultaten"
    ],
    "discussion": [
        r"discussion",
        r"discussie",
        r"beschouwing"
    ],
    "conclusion": [
        r"conclusions?",
        r"conclusies?",
        r"conclusie en aanbevelingen",
        r"conclusions? and recommendations?",
        r"aanbevelingen"
    ],
    "references": [
        r"references",
        r"bibliography",
        r"bibliografie",
        r"literatuur",
        r"literatuurlijst",
        r"bronnenlijst"
    ]
}

# Allow optional heading numbering, such as "1 Introduction" or "2.3 Methods"
NUMBER_PREFIX = r"(?:\d+(?:\.\d+)*\.?\s+)?"

heading_patterns = {
    section: re.compile(
        rf"^\s*{NUMBER_PREFIX}(?:{'|'.join(patterns)})\s*:?\s*$",
        flags=re.IGNORECASE
    )
    for section, patterns in SECTION_PATTERNS.items()
}

def detect_section_heading(line):
    """
    Identify a section heading from a complete text line.

    Returns a section label or None when no heading matches.
    """
    line = line.strip()

    if not line or len(line) > 100:
        return None

    for section, pattern in heading_patterns.items():
        if pattern.fullmatch(line):
            return section

    return None


# Test English and Dutch examples
test_headings = [
    "Abstract",
    "1 Introduction",
    "2.1 Materials and Methods",
    "3 Results",
    "4 Discussion",
    "5 Conclusions",
    "References",
    "Samenvatting",
    "1 Inleiding",
    "2 Onderzoeksopzet",
    "3 Resultaten",
    "4 Conclusie en aanbevelingen",
    "Literatuurlijst",
    "This paragraph discusses the results of the research."
]

for heading in test_headings:
    print(f"{heading:<55} -> {detect_section_heading(heading)}")

In [ ]:
section_records = []

for document_id, group in pages_df.groupby("document_id", sort=False):

    current_section = "unknown"
    section_number = 0

    for page in group.sort_values("page_number").itertuples(index=False):

        lines = page.original_text.splitlines(keepends=True)
        current_lines = []
        block_start_line = 1

        def save_block(lines_to_save, start_line, end_line):
            global section_number

            text = "".join(lines_to_save)

            if not text.strip():
                return

            section_number += 1

            section_records.append({
                "document_id": document_id,
                "file_name": page.file_name,
                "section_number": section_number,
                "section_label": current_section,
                "page_start": page.page_number,
                "page_end": page.page_number,
                "line_start": start_line,
                "line_end": end_line,
                "original_text": text,
                "character_count": len(text)
            })

        for line_number, line in enumerate(lines, start=1):

            detected_heading = detect_section_heading(line)

            if detected_heading is not None:

                # Save text preceding the heading
                save_block(
                    current_lines,
                    block_start_line,
                    line_number - 1
                )

                current_section = detected_heading
                current_lines = [line]
                block_start_line = line_number

            else:
                current_lines.append(line)

        # Save remaining text on the page
        save_block(
            current_lines,
            block_start_line,
            len(lines)
        )

sections_df = pd.DataFrame(section_records)

print("Total text blocks:", len(sections_df))
print("Documents with extracted text:", sections_df["document_id"].nunique())

display(
    sections_df[
        [
            "document_id",
            "section_number",
            "section_label",
            "page_start",
            "character_count"
        ]
    ].head(15)
)

In [ ]:
# Count blocks by section label
section_counts = (
    sections_df["section_label"]
    .value_counts()
    .rename_axis("section")
    .to_frame("blocks")
)

display(section_counts)

# Measure recognized versus unclassified text
total_characters = sections_df["character_count"].sum()

recognized_characters = sections_df.loc[
    sections_df["section_label"] != "unknown",
    "character_count"
].sum()

recognized_percentage = (
    100 * recognized_characters / total_characters
    if total_characters else 0
)

print(f"Total characters: {total_characters:,}")
print(f"Characters under recognized section labels: {recognized_percentage:.1f}%")
print(f"Unclassified characters: {100 - recognized_percentage:.1f}%")

# Check that segmentation preserves extracted page text
original_characters = pages_df["character_count"].sum()

assert total_characters == original_characters, (
    "Segmentation character count differs from extracted text."
)

print("Character preservation check: PASSED")

## 5. Segmentation quality assessment

Evaluate the initial section segmentation by examining the distribution of section labels and identifying potentially incorrect heading matches.

Compare extracted character counts with the original inventory and review examples of detected section boundaries.

These checks help identify extraction and segmentation limitations before language detection and further preprocessing.

In [ ]:
# Count characters assigned to each section label
section_characters = (
    sections_df.groupby("section_label")["character_count"]
    .sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.barh(
    section_characters.index,
    section_characters.values / 1_000_000,
    color=["#579AF5"]
)

ax.invert_yaxis()
ax.set_title("Extracted Text by Detected Section")
ax.set_xlabel("Characters (millions)")
ax.set_ylabel("Section label")

for bar in bars:
    ax.text(
        bar.get_width() + 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{bar.get_width():.2f}M",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
# Compare Notebook 01 character counts with Notebook 05
inventory_counts = documents_check[
    ["document_id", "file_name", "extracted_char_count"]
].copy()

inventory_counts["inventory_characters"] = pd.to_numeric(
    inventory_counts["extracted_char_count"],
    errors="coerce"
)

new_counts = (
    pages_df.groupby("document_id")["character_count"]
    .sum()
    .rename("preprocessing_characters")
)

comparison = inventory_counts.merge(
    new_counts,
    on="document_id",
    how="left"
)

comparison["preprocessing_characters"] = (
    comparison["preprocessing_characters"].fillna(0)
)

comparison["difference"] = (
    comparison["preprocessing_characters"]
    - comparison["inventory_characters"]
)

print("Inventory characters:",
      f"{comparison['inventory_characters'].sum():,.0f}")

print("Preprocessing characters:",
      f"{comparison['preprocessing_characters'].sum():,.0f}")

print("Difference:",
      f"{comparison['difference'].sum():,.0f}")

display(
    comparison[
        [
            "file_name",
            "inventory_characters",
            "preprocessing_characters",
            "difference"
        ]
    ]
    .assign(
        absolute_difference=lambda df: df["difference"].abs()
    )
    .sort_values("absolute_difference", ascending=False)
    .head(10)
)

In [ ]:
# Find blocks whose first line matches a section heading
heading_examples = sections_df.copy()

# Extract the first line of each text block
heading_examples["first_line"] = (
    heading_examples["original_text"]
    .str.split("\n")
    .str[0]
    .fillna("")
    .str.strip()
)

# Identify recognized section headings
heading_examples["detected_heading"] = (
    heading_examples["first_line"]
    .apply(detect_section_heading)
)

# Keep only blocks that begin with a recognized heading
heading_examples = heading_examples[
    heading_examples["detected_heading"].notna()
]

print("Detected heading occurrences:", len(heading_examples))

# Display a reproducible sample for manual inspection
display(
    heading_examples[
        [
            "file_name",
            "page_start",
            "detected_heading",
            "first_line"
        ]
    ].sample(
        n=min(20, len(heading_examples)),
        random_state=42
    )
)

### Extraction comparison

Notebook 01 extracted **6,684,765 characters**, while Notebook 05 extracted **9,367,556 characters** from the same 93 PDFs. This represents an increase of **2,682,791 characters (40.1%)**.

The notebooks use different PyMuPDF extraction settings: Notebook 01 uses the default text extraction, while Notebook 05 uses `sort=True` to improve reading order. This difference may contribute to the higher character count, although the exact cause has not been verified.

Notebook 05 retains its extraction method for subsequent segmentation and language detection. All **3,185 pages** were processed without extraction errors, and the segmentation character-preservation check passed.

The difference is recorded as a methodological limitation. The original inventory remains unchanged, and extraction completeness and reading-order accuracy have not yet been independently verified.

## 6. Improved section segmentation

Improve the initial segmentation by distinguishing front matter, tables of contents, and main document sections.

Use English and Dutch heading patterns together with page-level context to reduce incorrect section assignments.

Preserve the original text and document identifiers. Uncertain content remains unclassified for later review.

In [ ]:
# Additional patterns for front matter and tables of contents
FRONT_MATTER_PATTERNS = {
    "keywords": re.compile(
        r"^\s*(keywords?|trefwoorden|sleutelwoorden)\s*:?\s*$",
        re.IGNORECASE
    ),
    "contents": re.compile(
        r"^\s*(table of contents|contents|inhoudsopgave)\s*:?\s*$",
        re.IGNORECASE
    )
}

# Recognize a heading even when a PDF has trailing punctuation
def detect_extended_heading(line):
    """
    Detect scientific section headings and front-matter headings.
    """
    line = line.strip()

    if not line or len(line) > 100:
        return None

    for label, pattern in FRONT_MATTER_PATTERNS.items():
        if pattern.fullmatch(line):
            return label

    return detect_section_heading(line)


# Test additional heading patterns
test_headings = [
    "Table of Contents",
    "Inhoudsopgave",
    "Keywords",
    "Trefwoorden",
    "Abstract",
    "Samenvatting",
    "1 Introduction",
    "2 Methoden",
    "3 Results"
]

for heading in test_headings:
    print(f"{heading:<30} -> {detect_extended_heading(heading)}")

### 6.1 Table-of-contents identification

Identify possible table-of-contents pages using English and Dutch contents headings, page-number patterns, and repeated section entries.

Flag likely contents pages before segmentation to reduce false section boundaries.

These classifications are heuristic and may require manual verification.

In [ ]:
# Detect common table-of-contents entry formats
TOC_ENTRY_PATTERN = re.compile(
    r"^\s*(?:\d+(?:\.\d+)*\.?\s+)?"
    r".{3,90}?"
    r"(?:\.{2,}\s*|\s{2,})\d{1,4}\s*$"
)

def is_likely_contents_page(text):
    """
    Identify likely table-of-contents pages.

    Uses explicit contents headings and page-numbered entries.
    """
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if not lines:
        return False

    opening_lines = lines[:25]

    has_contents_heading = any(
        detect_extended_heading(line) == "contents"
        for line in opening_lines
    )

    numbered_entries = sum(
        bool(TOC_ENTRY_PATTERN.match(line))
        for line in lines
    )

    return (
        has_contents_heading
        or numbered_entries >= 5
    )


# Add contents-page flag
pages_df["is_contents_page"] = (
    pages_df["original_text"]
    .apply(is_likely_contents_page)
)

print(
    "Potential table-of-contents pages:",
    int(pages_df["is_contents_page"].sum())
)

print(
    "Documents with potential contents pages:",
    pages_df.loc[
        pages_df["is_contents_page"],
        "document_id"
    ].nunique()
)

display(
    pages_df.loc[
        pages_df["is_contents_page"],
        [
            "file_name",
            "page_number",
            "character_count"
        ]
    ].head(15)
)

### 6.2 Front-matter-aware segmentation

Rebuild the section labels using the identified contents pages and extended heading patterns.

Classify text before the first main section as front matter and retain abstracts and keywords as separate protected sections.

Preserve the extracted text exactly and verify that no characters are lost during segmentation.

In [ ]:
MAIN_SECTION_LABELS = {
    "introduction",
    "methods",
    "results",
    "discussion",
    "conclusion",
    "references"
}

PROTECTED_SECTION_LABELS = {
    "front_matter",
    "abstract",
    "keywords",
    "contents"
}

improved_records = []

for document_id, group in pages_df.groupby(
    "document_id",
    sort=False
):

    current_section = "front_matter"
    entered_main_body = False
    section_number = 0

    for page in group.sort_values("page_number").itertuples(index=False):

        # Keep likely contents pages separate
        if page.is_contents_page:
            if page.original_text:
                section_number += 1

                improved_records.append({
                    "document_id": document_id,
                    "file_name": page.file_name,
                    "section_number": section_number,
                    "section_label": "contents",
                    "page_start": page.page_number,
                    "page_end": page.page_number,
                    "original_text": page.original_text,
                    "character_count": len(page.original_text)
                })

            continue

        current_lines = []

        def save_improved_block(text_lines, label):
            global section_number

            text = "".join(text_lines)

            if not text:
                return

            section_number += 1

            improved_records.append({
                "document_id": document_id,
                "file_name": page.file_name,
                "section_number": section_number,
                "section_label": label,
                "page_start": page.page_number,
                "page_end": page.page_number,
                "original_text": text,
                "character_count": len(text)
            })

        for line in page.original_text.splitlines(keepends=True):

            heading = detect_extended_heading(line)

            if heading in MAIN_SECTION_LABELS:
                save_improved_block(current_lines, current_section)
                current_lines = []

                current_section = heading
                entered_main_body = True

            elif heading in {"abstract", "keywords"} and not entered_main_body:
                save_improved_block(current_lines, current_section)
                current_lines = []

                current_section = heading

            else:
                # Keep existing section label
                pass

            current_lines.append(line)

        save_improved_block(current_lines, current_section)

improved_sections_df = pd.DataFrame(improved_records)

print("Improved text blocks:", len(improved_sections_df))
print(
    "Documents represented:",
    improved_sections_df["document_id"].nunique()
)

display(
    improved_sections_df[
        [
            "document_id",
            "section_number",
            "section_label",
            "page_start",
            "character_count"
        ]
    ].head(15)
)

### 6.3 Segmentation validation

Check that improved segmentation preserves all extracted characters and document relationships.

Summarize the resulting section labels and identify how much text remains classified as front matter or other protected content.

These checks assess consistency and coverage, not the accuracy of individual section boundaries.

In [ ]:
original_total = pages_df["character_count"].sum()
improved_total = improved_sections_df["character_count"].sum()

print("Original extracted characters:", f"{original_total:,}")
print("Improved segmented characters:", f"{improved_total:,}")

assert original_total == improved_total, (
    "Character preservation failed."
)

print("Character preservation: PASSED")

# Count text blocks by label
improved_counts = (
    improved_sections_df["section_label"]
    .value_counts()
    .rename_axis("section")
    .to_frame("blocks")
)

display(improved_counts)

# Summarize protected text
protected_chars = improved_sections_df.loc[
    improved_sections_df["section_label"].isin(
        PROTECTED_SECTION_LABELS
    ),
    "character_count"
].sum()

protected_percentage = (
    protected_chars / improved_total * 100
    if improved_total else 0
)

print(
    f"Protected text: {protected_percentage:.1f}%"
)

### 6.4 Table-of-contents refinement

Review the initial table-of-contents classifications to reduce false positives in scientific articles and reports.

Use explicit English and Dutch contents headings and restrict detection to the opening pages of each document.

Compare the refined classifications with the initial results before updating the segmentation.

In [ ]:
# Detect explicit English and Dutch contents headings
CONTENTS_HEADING_PATTERN = re.compile(
    r"^\s*(table of contents|contents|inhoudsopgave|inhoud)\s*$",
    re.IGNORECASE
)

def has_explicit_contents_heading(text):
    """
    Check for a standalone contents heading near the top of a page.
    """
    opening_lines = [
        line.strip()
        for line in text.splitlines()[:30]
        if line.strip()
    ]

    return any(
        CONTENTS_HEADING_PATTERN.fullmatch(line)
        for line in opening_lines
    )


# Preserve the original heuristic flags for comparison
pages_df["initial_contents_flag"] = pages_df["is_contents_page"]

# Apply the more conservative rule
pages_df["is_contents_page"] = pages_df.apply(
    lambda row: (
        row["page_number"] <= 10
        and has_explicit_contents_heading(row["original_text"])
    ),
    axis=1
)

print(
    "Initially flagged pages:",
    int(pages_df["initial_contents_flag"].sum())
)

print(
    "Refined contents pages:",
    int(pages_df["is_contents_page"].sum())
)

print(
    "Documents with refined contents pages:",
    pages_df.loc[
        pages_df["is_contents_page"],
        "document_id"
    ].nunique()
)

display(
    pages_df.loc[
        pages_df["is_contents_page"],
        ["file_name", "page_number", "character_count"]
    ].head(20)
)

In [ ]:
# Identify pages no longer classified as contents
removed_flags = pages_df[
    pages_df["initial_contents_flag"]
    & ~pages_df["is_contents_page"]
]

print("Pages no longer flagged:", len(removed_flags))

display(
    removed_flags[
        ["file_name", "page_number", "character_count"]
    ].head(15)
)

### 6.5 Refined segmentation validation

Recalculate section distributions after refining table-of-contents detection.

Compare the initial and refined segmentation, verify character preservation, and visualize the proportion of text assigned to each section.

The labels remain heuristic and will be reviewed before selective stop-word removal.

In [ ]:
# Compare the original and refined segmentation by character count
initial_chars = (
    sections_df.groupby("section_label")["character_count"].sum()
)

refined_chars = (
    improved_sections_df.groupby("section_label")["character_count"].sum()
)

section_comparison = pd.concat(
    [
        initial_chars.rename("initial_characters"),
        refined_chars.rename("refined_characters")
    ],
    axis=1
).fillna(0).astype(int)

section_comparison["difference"] = (
    section_comparison["refined_characters"]
    - section_comparison["initial_characters"]
)

display(section_comparison.sort_index())

# Confirm that all characters are still preserved
assert (
    improved_sections_df["character_count"].sum()
    == pages_df["character_count"].sum()
)

print("Character preservation: PASSED")

In [ ]:
section_characters = (
    improved_sections_df.groupby("section_label")["character_count"]
    .sum()
    .sort_values()
)

ax = section_characters.plot(
    kind="barh",
    figsize=(10, 5),
    color="#579AF5"
)

ax.set_title("Refined section distribution")
ax.set_xlabel("Extracted characters")
ax.set_ylabel("Section label")

plt.tight_layout()
plt.show()

### 6.6 Section boundary inspection

Inspect transitions from front matter to main document sections to identify possible misclassifications.

Review selected documents with large front-matter sections and check whether actual body headings are recognized.

This manual inspection helps assess segmentation quality before language detection and selective preprocessing.

In [ ]:
# Summarize front-matter characters by document
front_matter_review = (
    improved_sections_df[
        improved_sections_df["section_label"] == "front_matter"
    ]
    .groupby(["document_id", "file_name"], as_index=False)
    ["character_count"]
    .sum()
    .rename(columns={
        "character_count": "front_matter_characters"
    })
)

# Add total document characters
document_totals = (
    pages_df.groupby("document_id", as_index=False)
    ["character_count"]
    .sum()
    .rename(columns={
        "character_count": "total_characters"
    })
)

front_matter_review = front_matter_review.merge(
    document_totals,
    on="document_id",
    how="left"
)

front_matter_review["front_matter_percentage"] = (
    100
    * front_matter_review["front_matter_characters"]
    / front_matter_review["total_characters"]
)

display(
    front_matter_review.sort_values(
        "front_matter_characters",
        ascending=False
    ).head(10)
)

In [ ]:
# Select documents with the largest front-matter character counts
review_ids = (
    front_matter_review
    .sort_values("front_matter_characters", ascending=False)
    .head(5)["document_id"]
)

for document_id in review_ids:
    document_sections = (
        improved_sections_df[
            improved_sections_df["document_id"] == document_id
        ]
        .sort_values("section_number")
    )

    print("\n" + "=" * 65)
    print(document_sections["file_name"].iloc[0])

    print("\nFirst section blocks:")
    for row in document_sections.head(6).itertuples(index=False):
        preview = " ".join(row.original_text.split())[:180]

        print(
            f"Page {row.page_start:>3} | "
            f"{row.section_label:<15} | "
            f"{row.character_count:>6} chars"
        )
        print(" ", preview, "\n")

### 6.7 Handling uncertain sections

Distinguish likely front matter from unclassified text when section headings cannot be reliably detected.

Limit automatic front-matter labeling to the beginning of a document and retain uncertain text as `unknown`.

Preserve all extracted text and avoid applying selective stop-word removal to uncertain sections.

In [ ]:
MAIN_SECTION_LABELS = {
    "introduction",
    "methods",
    "results",
    "discussion",
    "conclusion",
    "references"
}

document_section_summary = (
    improved_sections_df.groupby("document_id")["section_label"]
    .apply(lambda labels: any(label in MAIN_SECTION_LABELS for label in labels))
    .rename("has_main_section")
    .reset_index()
)

print(
    "Documents with recognized main sections:",
    document_section_summary["has_main_section"].sum()
)

print(
    "Documents without recognized main sections:",
    (~document_section_summary["has_main_section"]).sum()
)

display(
    document_section_summary[
        ~document_section_summary["has_main_section"]
    ].merge(
        documents_check[["document_id", "file_name"]],
        on="document_id",
        how="left"
    )[["file_name"]].head(20)
)

In [ ]:
# Work on a copy so the earlier results remain available
reviewed_sections_df = improved_sections_df.copy()

no_main_section_ids = set(
    document_section_summary.loc[
        ~document_section_summary["has_main_section"],
        "document_id"
    ]
)

uncertain_mask = (
    reviewed_sections_df["document_id"].isin(no_main_section_ids)
    & (reviewed_sections_df["page_start"] > 1)
    & (reviewed_sections_df["section_label"] == "front_matter")
)

reviewed_sections_df.loc[
    uncertain_mask,
    "section_label"
] = "unknown"

print("Reclassified blocks:", uncertain_mask.sum())

display(
    reviewed_sections_df["section_label"]
    .value_counts()
    .rename_axis("section_label")
    .to_frame("blocks")
)

assert (
    reviewed_sections_df["character_count"].sum()
    == pages_df["character_count"].sum()
)

print("Character preservation: PASSED")

## 7. Language detection using fastText

Use a pretrained fastText model to identify languages in the extracted scientific documents.

Detect Dutch and English while retaining other language predictions and model confidence scores.

Run all predictions locally and preserve the original text for subsequent preprocessing and embeddings.

In [ ]:
%pip install fasttext

In [ ]:
import fasttext

print("fastText imported successfully!")

In [ ]:
from pathlib import Path

MODEL_PATH = Path("../models/lid.176.bin").resolve()

print("Model found:", MODEL_PATH.exists())
print("Model size (MB):", round(MODEL_PATH.stat().st_size / 1_000_000, 1) if MODEL_PATH.exists() else "N/A")

In [ ]:
import fasttext

language_model = fasttext.load_model(str(MODEL_PATH))

print("fastText language model loaded successfully!")

### 7.1 Test language detection

Test the pretrained fastText model on Dutch and English scientific text.

Inspect predicted language labels and scores before applying the model to the full corpus.

In [ ]:
def detect_fasttext_language(text, min_chars=100):
    """
    Detect language using fastText.
    Return the language code and prediction score.
    """
    if not isinstance(text, str) or not text.strip():
        return "unknown", 0.0

    clean_text = " ".join(text.split())

    if len(clean_text) < min_chars:
        return "uncertain", 0.0

    # Use batch prediction for NumPy 2.x compatibility
    labels, scores = language_model.predict([clean_text], k=1)

    language = labels[0][0].replace("__label__", "")
    score = float(scores[0][0])

    return language, score

In [ ]:
test_texts = {
    "English": (
        "This research investigates the effects of climate change "
        "on coastal ecosystems and evaluates possible adaptation "
        "strategies for regional environmental management."
    ),
    "Dutch": (
        "Dit onderzoek richt zich op de gevolgen van klimaatverandering "
        "voor kustgebieden en onderzoekt mogelijke maatregelen "
        "voor duurzaam regionaal waterbeheer."
    ),
    "Short": "Introduction"
}

for name, text in test_texts.items():
    language, score = detect_fasttext_language(text)

    print(
        f"{name:<10} -> "
        f"Language: {language:<10} "
        f"Score: {score:.3f}"
    )

### 7.2 Document-level language detection

Apply fastText to the original extracted text of each PDF to identify its primary language.

Record the predicted language and model score, and summarize the language distribution across the corpus.

Preserve the original text and flag uncertain predictions for later review.

In [ ]:
# Combine page text into one text per document
document_texts = (
    pages_df
    .sort_values(["document_id", "page_number"])
    .groupby(["document_id", "file_name"], as_index=False)
    .agg(original_text=("original_text", "".join))
)

# Apply fastText
document_texts[
    ["detected_language", "language_score"]
] = pd.DataFrame(
    document_texts["original_text"]
    .apply(detect_fasttext_language)
    .tolist(),
    index=document_texts.index
)

print("Documents analyzed:", len(document_texts))

display(
    document_texts["detected_language"]
    .value_counts()
    .rename_axis("language")
    .to_frame("documents")
)

# Inspect unexpected or low-scoring predictions
language_review = document_texts[
    (~document_texts["detected_language"].isin(["nl", "en"]))
    | (document_texts["language_score"] < 0.80)
]

print("Documents flagged for review:", len(language_review))

display(
    language_review[
        ["file_name", "detected_language", "language_score"]
    ].sort_values("language_score")
)

In [ ]:
language_counts = (
    document_texts["detected_language"]
    .value_counts()
)

ax = language_counts.plot(
    kind="bar",
    figsize=(7, 4),
    color=["#579AF5", "#57C9A0", "#A78BFA", "#F29B85"]
)

ax.set_title("Primary language of extracted PDF documents")
ax.set_xlabel("Detected language")
ax.set_ylabel("Number of documents")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### 7.3 Paragraph-level language detection

Split the original extracted text into paragraph-like units and apply fastText to each unit.

Retain language predictions and scores to identify possible Dutch-English language variation within documents.

Short text and uncertain predictions remain available for review.

In [ ]:
import re

paragraph_records = []

for page in pages_df.itertuples(index=False):

    # Split on blank lines; preserve original page text separately
    paragraphs = re.split(
        r"\n\s*\n",
        page.original_text
    )

    for paragraph_number, paragraph in enumerate(paragraphs, start=1):
        if not paragraph.strip():
            continue

        paragraph_records.append({
            "document_id": page.document_id,
            "file_name": page.file_name,
            "page_number": page.page_number,
            "paragraph_number": paragraph_number,
            "original_text": paragraph,
            "character_count": len(paragraph)
        })

paragraphs_df = pd.DataFrame(paragraph_records)

print("Paragraph-like units:", len(paragraphs_df))

display(
    paragraphs_df[
        ["file_name", "page_number", "paragraph_number", "character_count"]
    ].head(10)
)

In [ ]:
paragraphs_df[
    ["detected_language", "language_score"]
] = pd.DataFrame(
    paragraphs_df["original_text"]
    .apply(detect_fasttext_language)
    .tolist(),
    index=paragraphs_df.index
)

print("Paragraph-level language distribution:")

display(
    paragraphs_df["detected_language"]
    .value_counts()
    .rename_axis("language")
    .to_frame("paragraphs")
)

In [ ]:
paragraphs_df[
    ["detected_language", "language_score"]
] = pd.DataFrame(
    paragraphs_df["original_text"]
    .apply(detect_fasttext_language)
    .tolist(),
    index=paragraphs_df.index
)

print("Paragraph-level language distribution:")

display(
    paragraphs_df["detected_language"]
    .value_counts()
    .rename_axis("language")
    .to_frame("paragraphs")
)

In [ ]:
reliable_paragraphs = paragraphs_df[
    (paragraphs_df["detected_language"].isin(["en", "nl"]))
    & (paragraphs_df["language_score"] >= 0.80)
    & (paragraphs_df["character_count"] >= 100)
]

language_mix = (
    reliable_paragraphs
    .groupby(["document_id", "file_name", "detected_language"])
    ["character_count"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=["en", "nl"], fill_value=0)
    .reset_index()
)

language_mix["total_classified_chars"] = (
    language_mix["en"] + language_mix["nl"]
)

language_mix["english_percentage"] = (
    100 * language_mix["en"]
    / language_mix["total_classified_chars"]
)

language_mix["dutch_percentage"] = (
    100 * language_mix["nl"]
    / language_mix["total_classified_chars"]
)

# Screening rule: at least 10% of classified text in each language
language_mix["potentially_mixed"] = (
    (language_mix["english_percentage"] >= 10)
    & (language_mix["dutch_percentage"] >= 10)
)

print(
    "Potentially mixed-language documents:",
    language_mix["potentially_mixed"].sum()
)

display(
    language_mix[
        language_mix["potentially_mixed"]
    ][
        [
            "file_name",
            "english_percentage",
            "dutch_percentage",
            "total_classified_chars"
        ]
    ].sort_values("dutch_percentage", ascending=False)
)

### 7.4 Language detection quality assessment

Evaluate paragraph lengths, language scores, and unexpected language predictions.

Identify possible PDF extraction artifacts and assess whether the current paragraph boundaries are suitable for language detection.

Use these findings to refine mixed-language screening without changing the original text.

In [ ]:
# Summarize paragraph lengths
print("Paragraph length statistics:")

display(
    paragraphs_df["character_count"]
    .describe()
    .to_frame("characters")
)

short_paragraphs = (
    paragraphs_df["character_count"] < 100
).sum()

print(
    f"Paragraphs shorter than 100 characters: "
    f"{short_paragraphs:,} "
    f"({short_paragraphs / len(paragraphs_df) * 100:.1f}%)"
)

# Visualize paragraph lengths
ax = paragraphs_df["character_count"].clip(upper=1000).hist(
    bins=40,
    figsize=(9, 4),
    color="#579AF5"
)

ax.set_title("Length of extracted paragraph-like units")
ax.set_xlabel("Characters (values above 1,000 capped)")
ax.set_ylabel("Number of units")

plt.tight_layout()
plt.show()

In [ ]:
# Examine predictions other than English, Dutch, or uncertain
unexpected_languages = paragraphs_df[
    ~paragraphs_df["detected_language"].isin(
        ["en", "nl", "uncertain", "unknown"]
    )
].copy()

print("Unexpected-language predictions:", len(unexpected_languages))

# Sample predictions for manual inspection
sample_size = min(15, len(unexpected_languages))

if sample_size:
    review_sample = unexpected_languages.sample(
        n=sample_size,
        random_state=42
    ).copy()

    review_sample["text_preview"] = (
        review_sample["original_text"]
        .str.replace("\n", " ", regex=False)
        .str.slice(0, 180)
    )

    display(
        review_sample[
            [
                "file_name",
                "page_number",
                "detected_language",
                "language_score",
                "character_count",
                "text_preview"
            ]
        ]
    )

In [ ]:
# Show documents with the highest minority-language share
mix_review = language_mix.copy()

mix_review["minority_language_percentage"] = (
    mix_review[
        ["english_percentage", "dutch_percentage"]
    ].min(axis=1)
)

display(
    mix_review[
        [
            "file_name",
            "english_percentage",
            "dutch_percentage",
            "minority_language_percentage",
            "total_classified_chars"
        ]
    ]
    .sort_values(
        "minority_language_percentage",
        ascending=False
    )
    .head(15)
)

### 7.5 Language prediction quality control

Review fastText predictions using text length, predicted language, and model scores.

Distinguish usable Dutch and English predictions from short, low-scoring, or unexpected-language passages.

Preserve the original predictions and text for transparency and manual review.

In [ ]:
# Keep original predictions and add a separate quality label
paragraphs_df["language_quality"] = "review"

# Short paragraphs
normalized_lengths = (
    paragraphs_df["original_text"]
    .apply(lambda text: len(" ".join(text.split())))
)

paragraphs_df.loc[
    normalized_lengths < 100,
    "language_quality"
] = "too_short"

# Usable Dutch or English predictions
usable_mask = (
    paragraphs_df["detected_language"].isin(["en", "nl"])
    & (paragraphs_df["language_score"] >= 0.80)
    & (normalized_lengths >= 100)
)

paragraphs_df.loc[
    usable_mask,
    "language_quality"
] = "usable"

# Low-scoring predictions
low_score_mask = (
    (normalized_lengths >= 100)
    & (paragraphs_df["language_score"] < 0.80)
)

paragraphs_df.loc[
    low_score_mask,
    "language_quality"
] = "low_score"

# Other predicted languages
other_language_mask = (
    (normalized_lengths >= 100)
    & ~paragraphs_df["detected_language"].isin(
        ["en", "nl", "uncertain", "unknown"]
    )
)

paragraphs_df.loc[
    other_language_mask,
    "language_quality"
] = "other_language_review"

display(
    paragraphs_df["language_quality"]
    .value_counts()
    .rename_axis("quality")
    .to_frame("paragraphs")
)

In [ ]:
quality_counts = (
    paragraphs_df["language_quality"]
    .value_counts()
    .sort_values()
)

ax = quality_counts.plot(
    kind="barh",
    figsize=(9, 4),
    color="#579AF5"
)

ax.set_title("Paragraph-level language prediction quality")
ax.set_xlabel("Number of paragraph-like units")
ax.set_ylabel("Quality category")

plt.tight_layout()
plt.show()

## 8. Optional stop-word removal using spaCy

Use spaCy's Dutch and English stop-word resources for optional text filtering.

Preserve negations and original text, and exclude metadata, references, and uncertain sections from automatic filtering.

Keep filtered text separate from the original text used for EuroBERT embeddings.

In [ ]:
%pip install spacy

In [ ]:
import spacy

# Lightweight tokenizers and language-specific stop-word lists
nlp_en = spacy.blank("en")
nlp_nl = spacy.blank("nl")

# Protect negations and other meaning-changing terms
PROTECTED_WORDS = {
    "en": {"no", "not", "nor", "never", "without"},
    "nl": {"niet", "geen", "nooit", "zonder", "noch"}
}

for word in PROTECTED_WORDS["en"]:
    nlp_en.vocab[word].is_stop = False

for word in PROTECTED_WORDS["nl"]:
    nlp_nl.vocab[word].is_stop = False

print("English stop words:", len(nlp_en.Defaults.stop_words))
print("Dutch stop words:", len(nlp_nl.Defaults.stop_words))
print("spaCy resources initialized.")

In [ ]:
def remove_stopwords(text, language):
    """
    Optionally remove stop words while preserving negations.

    Returns filtered text without modifying the original.
    """
    if language not in {"en", "nl"}:
        return text

    nlp = nlp_en if language == "en" else nlp_nl
    protected = PROTECTED_WORDS[language]

    doc = nlp.make_doc(text)

    retained = [
        token.text
        for token in doc
        if (
            not token.is_stop
            or token.lower_ in protected
            or token.is_punct
        )
    ]

    return " ".join(retained)


examples = {
    "en": "This study does not show that the results are without limitations.",
    "nl": "Dit onderzoek laat niet zien dat er geen beperkingen zijn."
}

for language, text in examples.items():
    print(f"\nLanguage: {language}")
    print("Original:", text)
    print("Filtered:", remove_stopwords(text, language))

### 8.1 Selective stop-word filtering

Refine the spaCy filtering function to preserve negations, numbers, and punctuation.

Keep filtering optional and store its output separately from the original text.

Apply filtering only to eligible body text, never to protected metadata or uncertain passages.

In [ ]:
def remove_stopwords(text, language):
    """
    Return optionally filtered text without modifying the original.
    Preserve negations, numbers, and punctuation.
    """
    if not isinstance(text, str) or language not in {"en", "nl"}:
        return text

    nlp = nlp_en if language == "en" else nlp_nl
    protected = PROTECTED_WORDS[language]

    doc = nlp.make_doc(text)

    retained = [
        token.text_with_ws
        for token in doc
        if (
            not token.is_stop
            or token.lower_ in protected
            or token.like_num
            or token.is_punct
        )
    ]

    return "".join(retained).strip()

In [ ]:
for language, text in examples.items():
    filtered = remove_stopwords(text, language)

    print(f"\nLanguage: {language}")
    print("Original:", text)
    print("Filtered:", filtered)

    for negation in PROTECTED_WORDS[language]:
        if negation in text.lower().split():
            assert negation in filtered.lower().split(), (
                f"Negation lost: {negation}"
            )

print("\nNegation preservation checks passed.")

### 8.2 Section-based filtering rules

Restrict optional stop-word removal to clearly identified body sections.

Protect titles, authors, affiliations, abstracts, keywords, references, and uncertain sections.

Preserve original text for EuroBERT embeddings and keep filtering results separate.

In [ ]:
# Sections where optional stop-word removal is permitted
ELIGIBLE_SECTIONS = {
    "introduction",
    "methods",
    "results",
    "discussion",
    "conclusion"
}

# Sections that must remain unchanged
PROTECTED_SECTIONS = {
    "front_matter",
    "abstract",
    "keywords",
    "contents",
    "references",
    "unknown"
}

print("Eligible sections:", sorted(ELIGIBLE_SECTIONS))
print("Protected sections:", sorted(PROTECTED_SECTIONS))

# Check section labels in the segmented data
existing_sections = set(reviewed_sections_df["section_label"])

print("\nSections found in the corpus:")
print(sorted(existing_sections))

unhandled_sections = existing_sections - (
    ELIGIBLE_SECTIONS | PROTECTED_SECTIONS
)

print("\nUnhandled section labels:", sorted(unhandled_sections))

### 8.3 Apply section-aware stop-word filtering

Combine section labels and fastText language detection to identify eligible text for optional filtering.

Apply spaCy stop-word removal only to sufficiently reliable Dutch and English body sections, while preserving protected and uncertain sections.

Store filtered text separately from the original text used for EuroBERT embeddings.

In [ ]:
# Work on a copy to preserve the segmented dataset
filtered_sections_df = reviewed_sections_df.copy()

# Detect language on each section block
language_results = (
    filtered_sections_df["original_text"]
    .apply(detect_fasttext_language)
    .tolist()
)

filtered_sections_df[
    ["detected_language", "language_score"]
] = pd.DataFrame(
    language_results,
    index=filtered_sections_df.index
)

# Determine which blocks are eligible for filtering
filtered_sections_df["filter_eligible"] = (
    filtered_sections_df["section_label"].isin(ELIGIBLE_SECTIONS)
    & filtered_sections_df["detected_language"].isin(["en", "nl"])
    & (filtered_sections_df["language_score"] >= 0.80)
)

# Default: keep text unchanged
filtered_sections_df["filtered_text"] = (
    filtered_sections_df["original_text"]
)

# Filter only eligible blocks
eligible_mask = filtered_sections_df["filter_eligible"]

filtered_sections_df.loc[
    eligible_mask, "filtered_text"
] = filtered_sections_df.loc[eligible_mask].apply(
    lambda row: remove_stopwords(
        row["original_text"],
        row["detected_language"]
    ),
    axis=1
)

print("Total section blocks:", len(filtered_sections_df))
print("Eligible for filtering:", eligible_mask.sum())
print("Protected or uncertain:", (~eligible_mask).sum())

In [ ]:
# Original text must remain identical
assert filtered_sections_df["original_text"].equals(
    reviewed_sections_df["original_text"]
)

# Protected or uncertain blocks must remain unchanged
unchanged_mask = ~filtered_sections_df["filter_eligible"]

assert (
    filtered_sections_df.loc[unchanged_mask, "original_text"]
    == filtered_sections_df.loc[unchanged_mask, "filtered_text"]
).all()

# Check character reduction in eligible blocks
eligible_blocks = filtered_sections_df[
    filtered_sections_df["filter_eligible"]
]

original_chars = eligible_blocks["original_text"].str.len().sum()
filtered_chars = eligible_blocks["filtered_text"].str.len().sum()

print("Original text preservation: PASSED")
print("Protected text preservation: PASSED")
print("Original eligible characters:", f"{original_chars:,}")
print("Filtered eligible characters:", f"{filtered_chars:,}")

if original_chars:
    print(
        "Character reduction:",
        f"{100 * (1 - filtered_chars / original_chars):.1f}%"
    )

### 8.3.1 Visualize selective stop-word filtering

Visualize the proportion of eligible and protected section blocks and compare text length before and after filtering.

Use these results to assess the extent of optional preprocessing while confirming that original text remains available for EuroBERT embeddings.

In [ ]:
import matplotlib.pyplot as plt

# Calculate summary statistics
eligible_count = int(filtered_sections_df["filter_eligible"].sum())
protected_count = int((~filtered_sections_df["filter_eligible"]).sum())

eligible = filtered_sections_df[
    filtered_sections_df["filter_eligible"]
]

original_chars = eligible["original_text"].str.len().sum()
filtered_chars = eligible["filtered_text"].str.len().sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Chart 1: Filtered vs protected blocks
axes[0].bar(
    ["Eligible", "Protected / uncertain"],
    [eligible_count, protected_count],
    color=["#A78BFA", "#F29B85"]
)

axes[0].set_title("Section blocks by filtering eligibility")
axes[0].set_ylabel("Number of section blocks")

for i, value in enumerate([eligible_count, protected_count]):
    axes[0].text(
        i, value + 20, f"{value:,}",
        ha="center"
    )

# Chart 2: Original vs filtered character counts
axes[1].bar(
    ["Original", "Filtered"],
    [original_chars, filtered_chars],
    color=["#579AF5", "#57C9A0"]
)

axes[1].set_title("Character reduction in eligible sections")
axes[1].set_ylabel("Characters")

for i, value in enumerate([original_chars, filtered_chars]):
    axes[1].text(
        i, value + original_chars * 0.015,
        f"{value / 1_000_000:.2f}M",
        ha="center"
    )

reduction = 100 * (1 - filtered_chars / original_chars)

axes[1].text(
    0.5, 0.90,
    f"{reduction:.1f}% reduction",
    transform=axes[1].transAxes,
    ha="center",
    fontsize=11,
    fontweight="bold"
)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

### 8.4 Validate filtered text quality

Inspect representative English and Dutch sections to evaluate the effect of stop-word removal.

Check whether negations, numbers, and important scientific terminology remain intact.

Use the original text as the authoritative representation and retain filtered text only as an optional preprocessing output.

In [ ]:
# Select reproducible examples from each language
for language in ["en", "nl"]:
    candidates = filtered_sections_df[
        filtered_sections_df["filter_eligible"]
        & (filtered_sections_df["detected_language"] == language)
        & (filtered_sections_df["original_text"].str.len() >= 300)
    ]

    print(f"\n{'=' * 60}")
    print(f"Language: {language}")
    print(f"Available examples: {len(candidates)}")

    if candidates.empty:
        continue

    sample = candidates.sample(
        n=min(2, len(candidates)),
        random_state=42
    )

    for _, row in sample.iterrows():
        print(f"\nFile: {row['file_name']}")
        print(f"Section: {row['section_label']}")
        print(f"Language score: {row['language_score']:.3f}")

        print("\nORIGINAL:")
        print(row["original_text"][:500])

        print("\nFILTERED:")
        print(row["filtered_text"][:500])

        print("-" * 60)

### 8.5 Protect meaning-changing words

Extend stop-word protection to conditional, contrastive, and modal expressions that may affect scientific meaning.

Keep stop-word filtering optional and preserve original text for EuroBERT embeddings.

Treat filtered text as an experimental representation because PDF section labels may include headers, figures, and metadata.

In [ ]:
PROTECTED_WORDS["en"].update({
    "however", "although", "though", "despite",
    "if", "unless", "whether",
    "can", "could", "may", "might", "must",
    "should", "would", "cannot"
})

PROTECTED_WORDS["nl"].update({
    "maar", "echter", "hoewel", "ondanks",
    "als", "indien", "tenzij", "of",
    "kan", "kunnen", "kon", "konden",
    "moet", "moeten", "zou", "zouden",
    "wel", "juist"
})

# Update spaCy's token-level stop-word flags
for word in PROTECTED_WORDS["en"]:
    nlp_en.vocab[word].is_stop = False

for word in PROTECTED_WORDS["nl"]:
    nlp_nl.vocab[word].is_stop = False

print("English protected words:", len(PROTECTED_WORDS["en"]))
print("Dutch protected words:", len(PROTECTED_WORDS["nl"]))

In [ ]:
test_sentences = {
    "en": (
        "However, despite this first sustainability wave, "
        "businesses may not be able to adapt."
    ),
    "nl": (
        "Als mijn bedrijf dat wil dan kan het "
        "duurzame energie toepassen."
    )
}

for language, original in test_sentences.items():
    filtered = remove_stopwords(original, language)

    print(f"\nLanguage: {language}")
    print("Original:", original)
    print("Filtered:", filtered)

    # Verify that protected words remain
    nlp = nlp_en if language == "en" else nlp_nl

    protected_in_original = {
        token.lower_
        for token in nlp.make_doc(original)
        if token.lower_ in PROTECTED_WORDS[language]
    }

    filtered_tokens = {
        token.lower_
        for token in nlp.make_doc(filtered)
    }

    assert protected_in_original.issubset(filtered_tokens), (
        f"Protected words lost: "
        f"{protected_in_original - filtered_tokens}"
    )

print("\nMeaning-changing word protection checks passed.")

### 8.6 Protect meaning-sensitive sentences

Preserve complete sentences containing conditional, contrastive, or modal expressions instead of filtering individual words within them.

Apply stop-word removal only to other eligible text and retain the original representation for EuroBERT embeddings.

In [ ]:
MEANING_SENSITIVE_WORDS = {
    "en": {
        "however", "although", "though", "despite",
        "if", "unless", "whether",
        "can", "could", "may", "might",
        "must", "should", "would", "not", "never"
    },
    "nl": {
        "maar", "echter", "hoewel", "ondanks",
        "als", "indien", "tenzij",
        "kan", "kunnen", "kon", "konden",
        "moet", "moeten", "zou", "zouden",
        "niet", "geen", "nooit"
    }
}

def remove_stopwords_conservative(text, language):
    if not isinstance(text, str) or language not in {"en", "nl"}:
        return text

    nlp = nlp_en if language == "en" else nlp_nl
    doc = nlp.make_doc(text)

    # Preserve the entire block if it contains meaning-sensitive terms.
    # This deliberately favors preservation over aggressive filtering.
    if any(
        token.lower_ in MEANING_SENSITIVE_WORDS[language]
        for token in doc
    ):
        return text

    return remove_stopwords(text, language)


for language, original in test_sentences.items():
    filtered = remove_stopwords_conservative(original, language)

    print(f"\nLanguage: {language}")
    print("Original:", original)
    print("Filtered:", filtered)

### 8.6.1 Evaluate conservative filtering coverage

Measure how many eligible section blocks contain meaning-sensitive expressions and would remain unchanged.

Compare protected and filterable blocks before deciding whether conservative filtering is useful for the corpus.

In [ ]:
# Start with the blocks eligible under Section 8.3
eligible_blocks = filtered_sections_df[
    filtered_sections_df["filter_eligible"]
].copy()

def contains_meaning_sensitive_words(text, language):
    if language not in {"en", "nl"}:
        return False

    nlp = nlp_en if language == "en" else nlp_nl

    return any(
        token.lower_ in MEANING_SENSITIVE_WORDS[language]
        for token in nlp.make_doc(text)
    )

eligible_blocks["meaning_sensitive"] = eligible_blocks.apply(
    lambda row: contains_meaning_sensitive_words(
        row["original_text"],
        row["detected_language"]
    ),
    axis=1
)

protected_count = int(eligible_blocks["meaning_sensitive"].sum())
filterable_count = int((~eligible_blocks["meaning_sensitive"]).sum())

print("Previously eligible blocks:", len(eligible_blocks))
print("Protected by meaning-sensitive rule:", protected_count)
print("Still filterable:", filterable_count)

print(
    "Still filterable (%):",
    f"{100 * filterable_count / len(eligible_blocks):.1f}%"
)

### 8.7 Sentence-level stop-word protection

Apply conservative stop-word filtering at sentence level rather than protecting entire section blocks.

Preserve sentences containing meaning-sensitive expressions while allowing other sentences to be filtered.

Keep all original text unchanged and use filtered text only as an optional representation.

In [ ]:
import re

def remove_stopwords_by_sentence(text, language):
    """
    Preserve meaning-sensitive sentences and optionally
    filter other sentences.
    """
    if not isinstance(text, str) or language not in {"en", "nl"}:
        return text

    # Lightweight sentence splitting without downloading spaCy models.
    # This is approximate and may split incorrectly at abbreviations.
    sentence_parts = re.split(
        r'(?<=[.!?])(\s+)',
        text
    )

    result = []

    for part in sentence_parts:
        if not part or part.isspace():
            result.append(part)
            continue

        nlp = nlp_en if language == "en" else nlp_nl
        doc = nlp.make_doc(part)

        contains_sensitive = any(
            token.lower_ in MEANING_SENSITIVE_WORDS[language]
            for token in doc
        )

        if contains_sensitive:
            result.append(part)
        else:
            result.append(remove_stopwords(part, language))

    return "".join(result)

In [ ]:
test_passages = {
    "en": (
        "This study examines sustainable business models. "
        "However, businesses may not be able to adapt. "
        "The research identifies several opportunities."
    ),
    "nl": (
        "Dit onderzoek beschrijft duurzame energie. "
        "Als mijn bedrijf dat wil dan kan het duurzame energie toepassen. "
        "De resultaten tonen verschillende mogelijkheden."
    )
}

for language, original in test_passages.items():
    filtered = remove_stopwords_by_sentence(original, language)

    print(f"\nLanguage: {language}")
    print("Original:", original)
    print("Filtered:", filtered)

### 8.8 Evaluate sentence-level filtering

Compare the original text, basic stop-word filtering, and sentence-level filtering on selected Dutch and English sections.

Inspect whether meaning-sensitive sentences remain intact and whether PDF formatting introduces filtering errors.

Use the results to decide whether the revised method is suitable for optional corpus preprocessing.

In [ ]:
# Select one reproducible section from each language
for language in ["en", "nl"]:
    candidates = filtered_sections_df[
        filtered_sections_df["filter_eligible"]
        & (filtered_sections_df["detected_language"] == language)
        & filtered_sections_df["original_text"].str.len().between(300, 2000)
    ]

    print(f"\n{'=' * 60}")
    print(f"Language: {language}")
    print(f"Available sections: {len(candidates)}")

    if candidates.empty:
        continue

    row = candidates.sample(n=1, random_state=42).iloc[0]

    original = row["original_text"]
    basic_filtered = remove_stopwords(
        original, language
    )
    sentence_filtered = remove_stopwords_by_sentence(
        original, language
    )

    print(f"\nFile: {row['file_name']}")
    print(f"Section: {row['section_label']}")

    print("\nORIGINAL:")
    print(original[:700])

    print("\nBASIC FILTERING:")
    print(basic_filtered[:700])

    print("\nSENTENCE-LEVEL FILTERING:")
    print(sentence_filtered[:700])

    print("\nCharacter reduction:")
    print(
        f"Basic: {100 * (1 - len(basic_filtered) / len(original)):.1f}%"
    )
    print(
        f"Sentence-level: "
        f"{100 * (1 - len(sentence_filtered) / len(original)):.1f}%"
    )

### 8.9 Identify structured text before filtering

Identify section blocks containing figures, tables, and other structured PDF content that may be damaged by stop-word removal.

Use conservative structural indicators to flag blocks for review while preserving original text.

Evaluate filtering coverage before deciding whether to apply additional safeguards.

In [ ]:
import re

STRUCTURED_TEXT_PATTERNS = [
    r"\bfigure\s+\d+",
    r"\bfig\.\s*\d+",
    r"\bfiguur\s+\d+",
    r"\btable\s+\d+",
    r"\btabel\s+\d+",
]

structured_pattern = re.compile(
    "|".join(STRUCTURED_TEXT_PATTERNS),
    flags=re.IGNORECASE
)

# Evaluate only previously eligible blocks
structure_review_df = filtered_sections_df[
    filtered_sections_df["filter_eligible"]
].copy()

structure_review_df["contains_figure_or_table"] = (
    structure_review_df["original_text"]
    .str.contains(structured_pattern, na=False)
)

# Count lines with large whitespace gaps (common in extracted tables)
def count_aligned_lines(text):
    return sum(
        bool(re.search(r"\S\s{5,}\S", line))
        for line in text.splitlines()
    )

structure_review_df["aligned_lines"] = (
    structure_review_df["original_text"]
    .apply(count_aligned_lines)
)

structure_review_df["likely_structured"] = (
    structure_review_df["contains_figure_or_table"]
    | (structure_review_df["aligned_lines"] >= 3)
)

print("Previously eligible blocks:", len(structure_review_df))
print(
    "Potentially structured blocks:",
    int(structure_review_df["likely_structured"].sum())
)
print(
    "Remaining candidates:",
    int((~structure_review_df["likely_structured"]).sum())
)

### 8.10 Evaluate filtering safeguards

Summarize the coverage of language-based eligibility, meaning-sensitive protection, and structural screening.

Document the limitations of heuristic stop-word removal and retain original text as the primary representation for downstream embeddings.

In [ ]:
total_blocks = len(filtered_sections_df)

basic_eligible = int(
    filtered_sections_df["filter_eligible"].sum()
)

meaning_safe = int(
    (~eligible_blocks["meaning_sensitive"]).sum()
)

structure_safe = int(
    (~structure_review_df["likely_structured"]).sum()
)

# Compare rules on the same section blocks
combined_review = eligible_blocks[
    ["document_id", "section_number", "meaning_sensitive"]
].merge(
    structure_review_df[
        ["document_id", "section_number", "likely_structured"]
    ],
    on=["document_id", "section_number"],
    validate="one_to_one"
)

combined_safe = int((
    ~combined_review["meaning_sensitive"]
    & ~combined_review["likely_structured"]
).sum())

safeguard_summary = pd.DataFrame({
    "Filtering strategy": [
        "Language + section eligibility",
        "Exclude meaning-sensitive blocks",
        "Exclude potentially structured blocks",
        "Apply both exclusions"
    ],
    "Remaining blocks": [
        basic_eligible,
        meaning_safe,
        structure_safe,
        combined_safe
    ]
})

safeguard_summary["Percentage of all blocks"] = (
    100 * safeguard_summary["Remaining blocks"] / total_blocks
).round(1)

display(safeguard_summary)

### 8.10.1 Visualize filtering safeguard coverage

Compare the number of section blocks remaining eligible under each filtering strategy.

Visualize how meaning-sensitive and structural safeguards reduce filtering coverage, supporting the decision to keep stop-word removal optional.

In [ ]:
import matplotlib.pyplot as plt

plot_df = safeguard_summary.copy()

colors = [
    "#579AF5",  # Blue: basic eligibility
    "#57C9A0",  # Green: meaning-sensitive protection
    "#A78BFA",  # Purple: structural protection
    "#F29B85"   # Coral: combined safeguards
]

fig, ax = plt.subplots(figsize=(10, 4.5))

bars = ax.barh(
    plot_df["Filtering strategy"],
    plot_df["Percentage of all blocks"],
    color=colors
)

# Display percentage and number of blocks
for bar, (_, row) in zip(bars, plot_df.iterrows()):
    ax.text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{row['Percentage of all blocks']:.1f}% "
        f"({int(row['Remaining blocks']):,} blocks)",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Filtering coverage under different safeguards",
    fontsize=13
)

ax.set_xlabel("Percentage of all section blocks")
ax.set_xlim(0, 65)
ax.invert_yaxis()

# Clean up chart borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### 8.11 Final stop-word filtering decision

Stop-word removal remains optional because it can alter scientific meaning and affect metadata, figures, and tables.

Applying both block-level safeguards leaves only 80 of 3,327 section blocks (2.4%) eligible, making this approach too restrictive for general preprocessing.

Original text is preserved for EuroBERT embeddings, while stop-word-filtered text is retained only as an experimental representation.

## 9. Final preprocessing validation and export

Validate document coverage, section segmentation, language detection, and text preservation before exporting the preprocessing results.

Ensure that all 93 PDFs are accounted for and that original text remains unchanged for EuroBERT embeddings.

Export validated results separately from optional stop-word-filtered text.

### 9.1 Validate corpus and segmentation consistency

Check document coverage, extracted character counts, and section labels against the latest contents-page detection.

Identify any inconsistencies before exporting the final preprocessing files.

In [ ]:
# 1. Document coverage
inventory_ids = set(documents_check["document_id"])
page_ids = set(pages_df["document_id"])
section_ids = set(reviewed_sections_df["document_id"])

print("Inventory documents:", len(inventory_ids))
print("Documents in pages:", len(page_ids))
print("Documents in sections:", len(section_ids))

print("\nMissing from pages:", len(inventory_ids - page_ids))
print("Missing from sections:", len(inventory_ids - section_ids))

# 2. Original text preservation
page_chars = pages_df["original_text"].str.len().sum()
section_chars = reviewed_sections_df["original_text"].str.len().sum()

print("\nPage characters:", f"{page_chars:,}")
print("Section characters:", f"{section_chars:,}")
print("Character totals match:", page_chars == section_chars)

# 3. Contents-page consistency
flagged_contents_pages = int(pages_df["is_contents_page"].sum())

contents_blocks = int(
    (reviewed_sections_df["section_label"] == "contents").sum()
)

print("\nPages flagged as contents:", flagged_contents_pages)
print("Section blocks labeled contents:", contents_blocks)

# 4. Filtered dataset consistency
print(
    "\nFiltered section count matches:",
    len(filtered_sections_df) == len(reviewed_sections_df)
)

print(
    "Original text preserved in filtered dataset:",
    filtered_sections_df["original_text"].equals(
        reviewed_sections_df["original_text"]
    )
)

### 9.2 Validate contents-page alignment

Compare the latest contents-page flags with the pages assigned to contents sections.

Identify potential stale segmentation labels before finalizing the preprocessing outputs.

In [ ]:
# Pages currently flagged as contents
flagged_pages = set(
    zip(
        pages_df.loc[
            pages_df["is_contents_page"], "document_id"
        ],
        pages_df.loc[
            pages_df["is_contents_page"], "page_number"
        ]
    )
)

# All page ranges covered by contents-labeled section blocks
contents_sections = reviewed_sections_df[
    reviewed_sections_df["section_label"] == "contents"
]

contents_pages = set()

for row in contents_sections.itertuples(index=False):
    for page_number in range(row.page_start, row.page_end + 1):
        contents_pages.add((row.document_id, page_number))

print("Flagged contents pages:", len(flagged_pages))
print("Pages covered by contents sections:", len(contents_pages))

print(
    "Contents-section pages not flagged:",
    len(contents_pages - flagged_pages)
)

print(
    "Flagged pages not covered by contents sections:",
    len(flagged_pages - contents_pages)
)

# Show a few mismatches without printing full PDF text
mismatch_rows = []

for document_id, page_number in sorted(contents_pages - flagged_pages):
    matching = pages_df[
        (pages_df["document_id"] == document_id)
        & (pages_df["page_number"] == page_number)
    ]

    if not matching.empty:
        mismatch_rows.append({
            "file_name": matching.iloc[0]["file_name"],
            "page_number": page_number
        })

display(pd.DataFrame(mismatch_rows).head(15))

### 9.3 Resolve stale contents-section labels

The refined contents-page detector flags 46 pages, while the current segmentation still covers 143 contents-labeled pages.

Review the segmentation construction step and rebuild section labels using the latest contents-page flags before exporting the corpus.

In [ ]:
print("Improved segmentation available:",
      "improved_sections_df" in globals())

print("Reviewed segmentation available:",
      "reviewed_sections_df" in globals())

if "improved_sections_df" in globals():
    print(
        "Contents blocks in improved segmentation:",
        (improved_sections_df["section_label"] == "contents").sum()
    )

print(
    "Contents blocks in reviewed segmentation:",
    (reviewed_sections_df["section_label"] == "contents").sum()
)

print("\nRelevant segmentation functions:")
print([
    name for name, obj in globals().items()
    if callable(obj)
    and any(term in name.lower()
            for term in ["segment", "section", "contents"])
])

In [ ]:
# Check whether contents blocks span multiple pages
contents_check = reviewed_sections_df.loc[
    reviewed_sections_df["section_label"] == "contents",
    ["document_id", "file_name", "page_start", "page_end"]
].copy()

contents_check["pages_spanned"] = (
    contents_check["page_end"]
    - contents_check["page_start"]
    + 1
)

print("Contents blocks:", len(contents_check))
print(
    "Multi-page contents blocks:",
    (contents_check["pages_spanned"] > 1).sum()
)

print("\nPages spanned per block:")
print(contents_check["pages_spanned"].value_counts().sort_index())

print("\nSample contents blocks:")
display(contents_check.head(10))

### 9.3 Correct stale contents labels

Update outdated contents labels using the refined page-level detector.

Preserve the original text and section boundaries, and assign uncertain blocks to `unknown` rather than guessing their scientific section.

In [ ]:
# Preserve the existing segmentation as a backup
reviewed_sections_before_contents_fix = reviewed_sections_df.copy()

# Current contents-page identifiers
valid_contents_pages = set(
    zip(
        pages_df.loc[pages_df["is_contents_page"], "document_id"],
        pages_df.loc[pages_df["is_contents_page"], "page_number"]
    )
)

# Identify outdated contents labels
is_contents = reviewed_sections_df["section_label"].eq("contents")

is_valid_contents = pd.Series(
    [
        (doc_id, page_start) in valid_contents_pages
        for doc_id, page_start in zip(
            reviewed_sections_df["document_id"],
            reviewed_sections_df["page_start"]
        )
    ],
    index=reviewed_sections_df.index
)

stale_contents = is_contents & ~is_valid_contents

# Relabel without modifying original text
reviewed_sections_df.loc[stale_contents, "section_label"] = "unknown"

print("Corrected outdated contents blocks:", stale_contents.sum())
print(
    "Remaining contents blocks:",
    reviewed_sections_df["section_label"].eq("contents").sum()
)

# Verify text preservation
assert reviewed_sections_df["original_text"].equals(
    reviewed_sections_before_contents_fix["original_text"]
)

assert (
    reviewed_sections_df["original_text"].str.len().sum()
    == pages_df["original_text"].str.len().sum()
)

print("Original text preservation: PASSED")

### 9.4 Refresh optional stop-word filtering

Regenerate the experimental filtered text using the corrected section labels and latest protected-word rules.

Keep original text unchanged and restrict filtering to eligible scientific sections with sufficiently confident English or Dutch language detection.

In [ ]:
# Start from the corrected segmentation
filtered_sections_df = reviewed_sections_df.copy()

# Detect language for each section
language_results = filtered_sections_df["original_text"].apply(
    detect_fasttext_language
)

filtered_sections_df["detected_language"] = language_results.apply(
    lambda result: result[0]
)

filtered_sections_df["language_score"] = language_results.apply(
    lambda result: result[1]
)

# Apply the existing eligibility rules
filtered_sections_df["filter_eligible"] = (
    filtered_sections_df["section_label"].isin(ELIGIBLE_SECTIONS)
    & filtered_sections_df["detected_language"].isin({"en", "nl"})
    & (filtered_sections_df["language_score"] >= 0.80)
)

# Preserve original text by default
filtered_sections_df["filtered_text"] = (
    filtered_sections_df["original_text"]
)

# Filter only eligible blocks
eligible_mask = filtered_sections_df["filter_eligible"]

filtered_sections_df.loc[eligible_mask, "filtered_text"] = (
    filtered_sections_df.loc[eligible_mask].apply(
        lambda row: remove_stopwords(
            row["original_text"],
            row["detected_language"]
        ),
        axis=1
    )
)

# Validate the refreshed dataset
assert filtered_sections_df["original_text"].equals(
    reviewed_sections_df["original_text"]
)

assert (
    filtered_sections_df.loc[
        ~eligible_mask, "filtered_text"
    ].equals(
        filtered_sections_df.loc[
            ~eligible_mask, "original_text"
        ]
    )
)

print("Total section blocks:", len(filtered_sections_df))
print("Eligible for filtering:", eligible_mask.sum())
print("Protected or uncertain:", (~eligible_mask).sum())
print("Original text preservation: PASSED")
print("Protected text preservation: PASSED")

### 9.5 Final text-preservation validation

Compare the original extracted text and segmented text for every document, including text order and character content.

Confirm that all text-bearing documents are represented and that optional filtering does not modify protected sections.

In [ ]:
# Reconstruct text for each document in page order
page_texts = (
    pages_df
    .sort_values(["document_id", "page_number"])
    .groupby("document_id")["original_text"]
    .agg("".join)
)

# Reconstruct text from section blocks in section order
section_texts = (
    reviewed_sections_df
    .sort_values(["document_id", "section_number"])
    .groupby("document_id")["original_text"]
    .agg("".join)
)

# Compare all documents with extracted text
text_bearing_ids = set(
    pages_df.loc[pages_df["has_text"], "document_id"]
)

missing_sections = text_bearing_ids - set(section_texts.index)

mismatched_documents = [
    doc_id
    for doc_id in text_bearing_ids & set(section_texts.index)
    if page_texts[doc_id] != section_texts[doc_id]
]

# Confirm the refreshed filtering dataset matches segmentation
section_columns = [
    "document_id",
    "section_number",
    "section_label",
    "original_text"
]

filtering_aligned = (
    reviewed_sections_df[section_columns]
    .reset_index(drop=True)
    .equals(
        filtered_sections_df[section_columns]
        .reset_index(drop=True)
    )
)

protected_mask = ~filtered_sections_df["filter_eligible"]

protected_unchanged = (
    filtered_sections_df.loc[protected_mask, "original_text"]
    .equals(
        filtered_sections_df.loc[protected_mask, "filtered_text"]
    )
)

print("Inventory documents:", len(documents_check))
print("Text-bearing documents:", len(text_bearing_ids))
print("Missing segmented documents:", len(missing_sections))
print("Documents with text mismatches:", len(mismatched_documents))
print("Filtering aligned with segmentation:", filtering_aligned)
print("Protected text unchanged:", protected_unchanged)

assert not missing_sections
assert not mismatched_documents
assert filtering_aligned
assert protected_unchanged

print("\nFINAL VALIDATION: PASSED")

### 9.6 Export validated preprocessing results

Export the validated section segmentation and language predictions for downstream analysis.

Preserve original text for EuroBERT embeddings and save stop-word-filtered text separately as an optional experimental output.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("../output/preprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Primary segmentation output (original text)
sections_path = OUTPUT_DIR / "sections.jsonl"

reviewed_sections_df.to_json(
    sections_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# 2. Section-level language metadata
language_path = OUTPUT_DIR / "section_languages.csv"

language_columns = [
    "document_id",
    "file_name",
    "section_number",
    "section_label",
    "detected_language",
    "language_score"
]

filtered_sections_df[language_columns].to_csv(
    language_path,
    index=False
)

# 3. Experimental stop-word-filtered output
experimental_path = (
    OUTPUT_DIR / "sections_filtered_experimental.jsonl"
)

filtered_sections_df.to_json(
    experimental_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# Confirm exported files
for path in [
    sections_path,
    language_path,
    experimental_path
]:
    print(
        f"{path.name}: "
        f"{path.stat().st_size / (1024**2):.2f} MB"
    )

print("\nExport completed.")

### 9.7 Verify exported files

Reload the exported files and verify their record counts, section labels, and original text.

Confirm that the primary EuroBERT input matches the validated segmentation and that the experimental filtering output remains separate.

In [ ]:
# Reload exported files
exported_sections = pd.read_json(
    sections_path, lines=True
)

exported_languages = pd.read_csv(
    language_path
)

exported_experimental = pd.read_json(
    experimental_path, lines=True
)

# Validate record counts
assert len(exported_sections) == len(reviewed_sections_df)
assert len(exported_languages) == len(filtered_sections_df)
assert len(exported_experimental) == len(filtered_sections_df)

# Validate primary original text
assert exported_sections["original_text"].equals(
    reviewed_sections_df["original_text"].reset_index(drop=True)
)

# Validate section labels
assert exported_sections["section_label"].equals(
    reviewed_sections_df["section_label"].reset_index(drop=True)
)

# Validate experimental original and filtered text
for column in ["original_text", "filtered_text"]:
    assert exported_experimental[column].equals(
        filtered_sections_df[column].reset_index(drop=True)
    )

# Validate language metadata
assert exported_languages["detected_language"].tolist() == (
    filtered_sections_df["detected_language"].tolist()
)

print("Primary sections:", len(exported_sections))
print("Language records:", len(exported_languages))
print("Experimental sections:", len(exported_experimental))
print("Original text preserved:", True)
print("Section labels preserved:", True)
print("Experimental text preserved:", True)
print("\nEXPORT VERIFICATION: PASSED")

### 9.8 Final preprocessing summary

The pipeline processed 93 PDFs, including 92 documents with extractable text, and produced 3,327 section blocks. All 9,367,556 extracted characters were preserved, and the exported files passed validation.

Section segmentation and language metadata are available for downstream analysis. Original text remains the primary input for EuroBERT, while stop-word-filtered text is stored separately as an optional experimental representation.

**Limitations:** Section detection is heuristic, some blocks remain labeled `unknown`, and language predictions may be unreliable for short or structured text.